In [1]:
import pandas as pd
import numpy as np
import joblib

### wczytywanie danych dla modelu Supervised Learning
df = joblib.load('df_with_all_clusters.joblib')

umap_embeddings = np.load('umap_embeddings_10d.npy')

print("Obiekty wczytane.")

### definicja Zmiennych 
#### cechy (features) na których będziemy trenować model
X = umap_embeddings

#### cel (target)
y = df['nmf_topic']

print(f"Kształt danych X (cechy): {X.shape}")

print(f"Kształt danych y (cel): {y.shape}")

Obiekty wczytane.
Kształt danych X (cechy): (850367, 10)
Kształt danych y (cel): (850367,)


In [3]:
from sklearn.model_selection import train_test_split

### podział na zbiór treningowy i testowy ---
###### dane w proporcji 80% na trening, 20% na test
###### random_state=42 - podział będzie zawsze taki sam
###### stratify=y - tutaj ważny parametr -> w zbiorze treningowym i testowym musi być taki sam % udział każdej z 20 klas / tematów
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Rozmiar X_train: {X_train.shape}")
print(f"Rozmiar X_test: {X_test.shape}")

Rozmiar X_train: (680293, 10)
Rozmiar X_test: (170074, 10)


In [5]:
### adding decision boundary visualisation function

from sklearn.decomposition import PCA
from sklearn.base import clone
from numpy import where, arange, meshgrid, hstack, unique
import matplotlib.pyplot as plt

def visualize_decision_boundary_multiclass(X_data, y_data, trained_classifier, title="Granica decyzyjna modelu"):
    """
    Wizualizuje granicę decyzyjną klasyfikatora dla wielu klas 
    po zredukowaniu danych do 2 wymiarów za pomocą PCA.

    Argumenty:
    - X_data: Oryginalne, przetworzone dane (np. X_train_prepared).
    - y_data: Etykiety (np. y_train lub y_test).
    - trained_classifier: Model, którego typ chcemy zwizualizować (np. Random Forest itd.).
    - title: Tytuł wykresu.
    """
    ## redukcja wymiarów danych do 2D
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_data)

    ## klonowanie modelu z jego hiperparametrami aby zachować wszystkie ustawienia (np. n_estimators, max_depth, C, etc.)
    model_2d = clone(trained_classifier)
    model_2d.fit(X_pca, y_data)

    ## tworzenie siatki
    min1, max1 = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
    min2, max2 = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
    x1grid = arange(min1, max1, 0.1)
    x2grid = arange(min2, max2, 0.1)
    xx, yy = meshgrid(x1grid, x2grid)
    grid = hstack((xx.flatten().reshape(-1,1), yy.flatten().reshape(-1,1)))

    ## predykcja i rysowanie tła
    yhat = model_2d.predict(grid)
    zz = yhat.reshape(xx.shape)
    
    plt.figure(figsize=(16, 16))
    ## ta sama paleta co dla punktów, aby tło pasowało kolorami
    unique_classes = unique(y_data)
    cmap_background = plt.get_cmap('viridis', len(unique_classes))
    plt.contourf(xx, yy, zz, cmap=cmap_background, alpha=0.3)

    ## dynamiczne rysowanie punktów dla wielu klas
    cmap_points = plt.get_cmap('viridis', len(unique_classes))
    for i, class_value in enumerate(unique_classes):
        row_ix = where(y_data == class_value)
        plt.scatter(X_pca[row_ix, 0], X_pca[row_ix, 1],
                    color=cmap_points(i),
                    label=f'Klasa {class_value}',
                    alpha=0.7)

    plt.title(title, fontsize=15)
    plt.xlabel("Główna składowa 1 (PC1)", fontsize=12)
    plt.ylabel("Główna składowa 2 (PC2)", fontsize=12)
    plt.legend()
    plt.show()